# Modeling

## Setup

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

In [ ]:
seed = 42

## Load and inspect data

In [ ]:
train_test = gpd.read_file("../../../data/train_test.shp")
train_test.shape

In [ ]:
train_test.info()

In [ ]:
train_test.head()

In [ ]:
train_test["geometry"].drop_duplicates().plot()

In [ ]:
train_test.isna().sum()

Drop districts entirely if they don't have a complete timeseries, i. e. missing values for some ECV features?

In [ ]:
train_test = train_test.dropna().reset_index(drop=True)
train_test.shape

## Check feature distributions

In [ ]:
def plot_feature_distributions(data, columns):
    fig, axes = plt.subplots(1, len(columns), sharey=True, figsize=(9, 3))
    axes = axes.ravel()

    for col, ax in zip(columns, axes, strict=False):
        sns.histplot(
            data,
            x=col,
            hue="outbreak",
            multiple="dodge",
            stat="probability",
            common_norm=False,
            ax=ax,
        )

    fig.tight_layout()
    plt.show()

In [ ]:
plot_feature_distributions(train_test, ["sss", "sss_1m_l", "sss_2m_l"])

In [ ]:
plot_feature_distributions(train_test, ["sss_d_1m_l", "sss_d_2m_l"])

In [ ]:
plot_feature_distributions(train_test, ["chl", "chl_1m_l", "chl_2m_l"])

In [ ]:
plot_feature_distributions(train_test, ["chl_d_1m_l", "chl_d_2m_l"])

In [ ]:
plot_feature_distributions(train_test, ["lst", "lst_1m_l", "lst_2m_l"])

In [ ]:
plot_feature_distributions(train_test, ["lst_d_1m_l", "lst_d_2m_l"])

In [ ]:
plot_feature_distributions(train_test, ["west", "east"])

In [ ]:
plot_feature_distributions(train_test, ["mnsn", "post_mnsn", "pre_mnsn", "winter"])

## Train test split

In [ ]:
districts = train_test[["district"]]

In [ ]:
districts.nunique()

In [ ]:
districts.value_counts()

In [ ]:
X = train_test.drop(["state", "district", "year", "month", "outbreak", "geometry"], axis=1)

In [ ]:
y = train_test["outbreak"]

In [ ]:
y.value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)

## Train random forest

In [ ]:
rf = RandomForestClassifier(n_estimators=50, random_state=seed)

In [ ]:
%%time

rf.fit(X_train, y_train)

In [ ]:
y_pred = rf.predict(X_test)

## Evaluation

In [ ]:
print(confusion_matrix(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
y_pred_proba = rf.predict_proba(X_test)[:, 1]

In [ ]:
auc = roc_auc_score(y_test, y_pred_proba)

In [ ]:
auc

In [ ]:
baseline = [0 for _ in range(len(y_test))]
baseline_auc = roc_auc_score(y_test, baseline)

baseline_fpr, baseline_tpr, _ = roc_curve(y_test, baseline)
rf_fpr, rf_tpr, _ = roc_curve(y_test, y_pred_proba)

plt.plot(baseline_fpr, baseline_tpr, linestyle="--", label="Baseline")
plt.plot(rf_fpr, rf_tpr, marker=".", label="Random Forest")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.legend()

plt.show()

In [ ]:
feature_names = X.columns
importances = rf.feature_importances_
rf_importances = pd.Series(importances, index=feature_names).sort_values()

In [ ]:
fig, ax = plt.subplots()
rf_importances.plot.barh(ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_xlabel("Mean decrease in impurity")
fig.tight_layout()

## SMOTE

In [ ]:
oversample = SMOTE(sampling_strategy=0.1)
X_train, y_train = oversample.fit_resample(X_train, y_train)

In [ ]:
X_train.shape

In [ ]:
y_train.value_counts()

In [ ]:
rf = RandomForestClassifier(n_estimators=50, random_state=seed)

In [ ]:
%%time

rf.fit(X_train, y_train)

In [ ]:
y_pred = rf.predict(X_test)

In [ ]:
print(confusion_matrix(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
y_pred_proba = rf.predict_proba(X_test)[:, 1]

In [ ]:
auc = roc_auc_score(y_test, y_pred_proba)

In [ ]:
auc

In [ ]:
baseline = [0 for _ in range(len(y_test))]
baseline_auc = roc_auc_score(y_test, baseline)

baseline_fpr, baseline_tpr, _ = roc_curve(y_test, baseline)
rf_fpr, rf_tpr, _ = roc_curve(y_test, y_pred_proba)

plt.plot(baseline_fpr, baseline_tpr, linestyle="--", label="Baseline")
plt.plot(rf_fpr, rf_tpr, marker=".", label="Random Forest")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.legend()

plt.show()